# Per-forward error against fp32: Wan2.1-T2V-1.3B on L4

Runtime: L4. Predictions to check: `predictions_forward_error_2026-11-18.md` (H1-H10).

Every variant runs one transformer forward on the same captured inputs, and its output is compared with an fp32 forward on those inputs. Time: about 10 min to build SageAttention, then about 1 h 10 min of runs (~2.7 compute units). No cell waits for input; the last cell downloads the zip. Large tensors stay in /content/fe33 and /content/fe81, outside the zip.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,clocks.max.sm,power.limit,driver_version --format=csv

In [ ]:
!pip install -q diffusers==0.42.1 transformers==5.22.2 accelerate ftfy imageio imageio-ffmpeg lpips 2>&1 | tail -5
import torch, diffusers, transformers
print(torch.__version__, 'cuda', torch.version.cuda, diffusers.__version__, transformers.__version__)
!nvcc --version | tail -1

In [ ]:
%cd /content
!git -C video-dit-l4 pull || git clone https://github.com/FredWen-S/video-dit-l4.git
%cd /content/video-dit-l4

## Build SageAttention for sm_89 (~10 min)

Same build and commit as before. Stops the notebook if it fails.

In [ ]:
import os, re, subprocess, sys, torch
maj, mi = torch.version.cuda.split('.')[:2]
m = re.search(r'release (\d+)\.(\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
print('torch CUDA', torch.version.cuda, '| nvcc', m.group(0) if m else 'not found')
if not m or m.group(1) != maj:
    ubuntu = subprocess.run('. /etc/os-release && echo ubuntu${VERSION_ID/./}', shell=True, capture_output=True,
                            text=True, executable='/bin/bash').stdout.strip()
    v = f'{maj}-{mi}'
    !wget -q https://developer.download.nvidia.com/compute/cuda/repos/{ubuntu}/x86_64/cuda-keyring_1.1-1_all.deb && dpkg -i cuda-keyring_1.1-1_all.deb > /dev/null
    !apt-get -qq update > /dev/null && apt-get -qq install -y cuda-minimal-build-{v} cuda-cudart-dev-{v} cuda-cccl-{v} cuda-nvrtc-dev-{v} libcublas-dev-{v} libcusparse-dev-{v} libcusolver-dev-{v} libcurand-dev-{v} > /dev/null
    os.environ['CUDA_HOME'] = f'/usr/local/cuda-{maj}.{mi}'
    os.environ['PATH'] = os.environ['CUDA_HOME'] + '/bin:' + os.environ['PATH']
    !nvcc --version | tail -2

%cd /content
!git clone -q https://github.com/thu-ml/SageAttention.git || true
!git -C SageAttention checkout -q d1a57a5  # the commit measured in results/L4_sage.md
!git -C SageAttention log -1 --format='SageAttention %h %cd'
%cd /content/SageAttention
# torch >= 2.10 headers use C++20 `requires` clauses; SageAttention's setup.py hardcodes -std=c++17
!sed -i 's/-std=c++17/-std=c++20/g' setup.py && grep -n 'std=c++' setup.py && g++ --version | head -1
!TORCH_CUDA_ARCH_LIST=8.9 EXT_PARALLEL=4 MAX_JOBS=8 pip install --no-build-isolation -v . > /content/sage_build.log 2>&1; tail -40 /content/sage_build.log
%cd /content/video-dit-l4
r = subprocess.run([sys.executable, '-c', 'import importlib.metadata as m, sageattention; print("sageattention", m.version("sageattention"))'],
                   capture_output=True, text=True)
print(r.stdout, r.stderr[-2000:])
if r.returncode != 0:
    raise RuntimeError('SageAttention did not build or import; stopping here. Send the output above.')

## 33 frames, 4 prompts

Capture the base trajectory's transformer inputs at steps 0, 10, 20, 30, 40, 49, both CFG branches: 48 inputs (~8 min).

In [ ]:
!mkdir -p results/h_logs
!python bench/forward_error.py capture --frames 33 --steps 0 10 20 30 40 49 --out /content/fe33/inputs.pt 2>&1 | tee results/h_logs/capture_33f.log

One process per variant (fp32 ~8 min, each bf16 variant ~3 min). H1: `base` must reproduce the captured outputs, and every repeat must be identical.

In [ ]:
for v in ['fp32', 'eager', 'base', 'fp8_row_fast', 'sage_fp8pv', 'compile', 'compile_emul', 'fp8_row', 'fp8_tensor']:
    !python bench/forward_error.py run --inputs /content/fe33/inputs.pt --variant {v} --out /content/fe33 2>&1 | grep -v Warning | tee results/h_logs/run_33f_{v}.log | tail -3

Errors against fp32 (H2-H9).

In [ ]:
!python bench/forward_error.py compare --dir /content/fe33 --report results/h_forward_error_33f.json 2>&1 | tee results/h_logs/compare_33f.log

## 81 frames, fox, steps 0, 20, 49 (~25 min), H10

In [ ]:
!python bench/forward_error.py capture --frames 81 --prompt-ids 0 --steps 0 20 49 --out /content/fe81/inputs.pt 2>&1 | tee results/h_logs/capture_81f.log
for v in ['fp32', 'eager', 'base', 'sage_fp8pv', 'fp8_row_fast']:
    !python bench/forward_error.py run --inputs /content/fe81/inputs.pt --variant {v} --repeat 2 --out /content/fe81 2>&1 | grep -v Warning | tee results/h_logs/run_81f_{v}.log | tail -3
!python bench/forward_error.py compare --dir /content/fe81 --report results/h_forward_error_81f.json 2>&1 | tee results/h_logs/compare_81f.log

In [ ]:
# Zip the reports and logs (no tensors)
!zip -qr fwderr_results.zip results/h_forward_error_*.json results/h_logs && ls -lh fwderr_results.zip
from google.colab import files; files.download('fwderr_results.zip')